# Audio Data Augmentation - Syllable Dataset Expansion

## 📋 Project Overview
Pipeline untuk augmentasi audio suku kata dengan pendekatan **2-Stage Consolidation**:
1. **STAGE 1 - CONSOLIDATION**: Baca SEMUA extracted_syllables folders, gabung ke satu folder utuh
2. **STAGE 2 - AUGMENTATION**: Augmentasi data gabungan sekali saja dengan variasi fonetik alami

### 📁 Module Structure
- **augmentation_config.py**: Konfigurasi augmentasi (pitch shift, time stretch, normalisasi)
- **audio_augmenter.py**: Fungsi utility untuk audio augmentation
- **augmentation_pipeline.py**: Main pipeline yang mengintegrasikan proses augmentasi
- **path_manager.py**: Utility untuk manajemen direktori dan run tracking

## 🔧 Setup & Dependencies
Mengimport semua library dan inisialisasi konfigurasi augmentasi dengan pendekatan **2-Stage Consolidation**.

### STAGE 1 - CONSOLIDATION
- **Auto-detect** SEMUA `extracted_syllables_[X]/` folders yang tersedia
- **Merge** semua files dari berbagai extracted_syllables ke satu folder: `extracted_syllables_consolidated/`
- Organized by label: a/, ba/, be/, ma/, dll
- **Persiapan**: Memastikan semua source data dalam satu tempat

### STAGE 2 - AUGMENTATION
- **Input**: `extracted_syllables_consolidated/` (data hasil merge)
- **Process**: Augmentasi dengan pitch shift, time stretch, normalisasi
- **Output**: `augmented_[Y]/` (next sequential index)
- **Hasil**: Data augmentasi terpusat yang siap untuk training

**Output yang Ditampilkan:**
- ✓ Konfirmasi import berhasil
- ⚠️ Info extracted_syllables folders yang tersedia (sumber untuk consolidation)
- 📊 Consolidation summary (files per label)
- 📈 Augmentation results (output dengan index sequential)
- ✔️ Variable `consolidated_result` dan `current_aug_index` untuk analisis lanjutan


In [1]:
import os
import sys
import re
import shutil
import pandas as pd
import numpy as np
from pathlib import Path

# Import augmentation libraries
from libs import (
    augmentation_config,
    augmentation_pipeline, 
    path_manager,
    audio_augmenter,
    config
)

c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Checking Available extracted_syllables Folders (Input Data untuk Consolidation)
Menampilkan daftar SEMUA extracted_syllables folder yang akan dikonsolidasikan menjadi satu folder gabungan.

**ℹ️ Input Struktur dari Scraping:**
- `extracted_syllables_[X]/` → Extracted syllables folder dengan independent index X
  - Berisi label subdirectories: a/, ba/, be/, bu/, dll
  - Setiap label folder berisi audio files yang sudah diekstraksi dari berbagai scraper run

**Helper Functions:**
- `get_all_extracted_syllables_indices()` → List semua extracted_syllables indices [1,2,5,...]
- `consolidate_all_extracted_syllables()` → Merge semua ke extracted_syllables_consolidated/
- `get_next_augmented_index()` → Hitung next sequential index untuk output


In [2]:
# Check available EXTRACTED_SYLLABLES FOLDERS (input data untuk consolidation)
from libs import config
from libs.path_manager import get_run_info
import re

# ========== HELPER 1: Get All extracted_syllables Indices ==========
def get_all_extracted_syllables_indices(base_dir: str = config.BASE_SCRAPED_DIR) -> list:
    """
    Dapatkan SEMUA extracted_syllables_[X] indices yang tersedia
    
    Returns:
        List sorted dari extracted_syllables indices [1, 2, 3, 5, ...]
        
    Example:
        >>> get_all_extracted_syllables_indices()
        [1, 2, 3, 5]
    """
    if not os.path.exists(base_dir):
        return []
    
    indices = []
    for item in os.listdir(base_dir):
        if item.startswith("extracted_syllables_") and os.path.isdir(os.path.join(base_dir, item)):
            # Skip consolidated folder
            if item == "extracted_syllables_consolidated":
                continue
            
            match = re.match(r"^extracted_syllables_(\d+)$", item)
            if match:
                try:
                    idx = int(match.group(1))
                    indices.append(idx)
                except (ValueError, IndexError):
                    continue
    
    return sorted(indices) if indices else []


# ========== HELPER 2: Consolidate ALL extracted_syllables ==========
def consolidate_all_extracted_syllables(
    base_dir: str = config.BASE_SCRAPED_DIR,
    target_labels: list = None,
    consolidated_folder_name: str = "extracted_syllables_consolidated"
) -> dict:
    """
    CONSOLIDATE (merge) SEMUA extracted_syllables_[X] folders menjadi SATU folder gabungan.
    
    This function:
    1. Scans all extracted_syllables_[X] folders
    2. Creates extracted_syllables_consolidated/
    3. Copies all audio files dari setiap label
    4. Preserves label directory structure
    
    Args:
        base_dir: Base directory (BASE_SCRAPED_DIR)
        target_labels: List of target labels (dari augmentation_config.TARGET_LABELS)
        consolidated_folder_name: Nama folder hasil consolidation
    
    Returns:
        Dict dengan:
        - consolidated_path: Path ke folder hasil consolidation
        - input_indices: List of source extracted_syllables indices
        - label_summary: Dict {label: file_count}
        - total_files: Total files diconsolidate
        - status: 'success' atau 'error'
    
    Example:
        >>> result = consolidate_all_extracted_syllables()
        >>> print(result['total_files'])
        330
    """
    if target_labels is None:
        target_labels = augmentation_config.TARGET_LABELS
    
    # Get all extracted_syllables indices
    input_indices = get_all_extracted_syllables_indices(base_dir)
    
    if len(input_indices) == 0:
        return {
            "status": "error",
            "error": "No extracted_syllables folders found"
        }
    
    # Create consolidated folder
    consolidated_path = os.path.join(base_dir, consolidated_folder_name)
    
    # Clear existing consolidated folder jika ada
    if os.path.exists(consolidated_path):
        print(f"⚠️  Existing consolidated folder found → Clearing...")
        shutil.rmtree(consolidated_path)
    
    # Create structure
    for label in target_labels:
        label_dir = os.path.join(consolidated_path, label)
        os.makedirs(label_dir, exist_ok=True)
    
    # Copy files dari SEMUA extracted_syllables
    label_summary = {label: 0 for label in target_labels}
    
    for extracted_idx in input_indices:
        extracted_path = os.path.join(base_dir, f"extracted_syllables_{extracted_idx}")
        
        if not os.path.exists(extracted_path):
            continue
        
        # Copy untuk setiap label
        for label in target_labels:
            src_label_dir = os.path.join(extracted_path, label)
            dst_label_dir = os.path.join(consolidated_path, label)
            
            if os.path.exists(src_label_dir):
                for file in os.listdir(src_label_dir):
                    if file.endswith('.wav'):
                        src_file = os.path.join(src_label_dir, file)
                        dst_file = os.path.join(dst_label_dir, file)
                        
                        # Copy (overwrite jika ada duplicate)
                        shutil.copy2(src_file, dst_file)
                        label_summary[label] += 1
    
    total_files = sum(label_summary.values())
    
    return {
        "status": "success",
        "consolidated_path": consolidated_path,
        "input_indices": input_indices,
        "label_summary": label_summary,
        "total_files": total_files
    }


# ========== HELPER 3: Get Next Index untuk Augmented Outputs ==========
def get_next_augmented_index(base_dir: str = augmentation_config.BASE_AUGMENTED_DIR) -> int:
    """
    Hitung next SEQUENTIAL index untuk augmented folder (INDEPENDENT dari input index)
    
    Returns:
        Index berikutnya untuk augmented_[X]
        
    Example:
        Jika ada augmented_1, augmented_2 → return 3
    """
    if not os.path.exists(base_dir):
        return 1
    
    indexes = []
    for item in os.listdir(base_dir):
        match = re.match(r"^augmented_(\d+)$", item)
        if match:
            try:
                idx = int(match.group(1))
                indexes.append(idx)
            except (ValueError, IndexError):
                continue
    
    return max(indexes) + 1 if indexes else 1


# ========== NOW CALL THE FUNCTIONS ==========

print("\n" + "="*70)
print("📊 AVAILABLE INPUT DATA (extracted_syllables Folders)")
print("="*70)

extracted_indices = get_all_extracted_syllables_indices()

if len(extracted_indices) > 0:
    print(f"\n✓ Total extracted_syllables folders: {len(extracted_indices)}")
    print(f"✓ Indices available: {extracted_indices}")
    print(f"✓ These will be CONSOLIDATED into: extracted_syllables_consolidated/")
    
    # Show total files summary
    total_files_available = 0
    for extracted_idx in extracted_indices:
        extracted_path = os.path.join(config.BASE_SCRAPED_DIR, f"extracted_syllables_{extracted_idx}")
        file_count = 0
        if os.path.exists(extracted_path):
            for label in augmentation_config.TARGET_LABELS:
                label_path = os.path.join(extracted_path, label)
                if os.path.exists(label_path):
                    file_count += len([f for f in os.listdir(label_path) if f.endswith('.wav')])
        
        print(f"   - extracted_syllables_{extracted_idx}: {file_count} files")
        total_files_available += file_count
    
    print(f"\n✓ Total files to consolidate: {total_files_available}")
else:
    print("\n❌ No extracted_syllables folders found!")
    print("   Run scraping.ipynb terlebih dahulu untuk generate data")

print("="*70)
print("\n✓ Helper functions loaded:")
print("   - get_all_extracted_syllables_indices() → Get SEMUA extracted_syllables indices")
print("   - consolidate_all_extracted_syllables() → Merge semua ke satu folder")
print("   - get_next_augmented_index() → Get next SEQUENTIAL augmentation index")


📊 AVAILABLE INPUT DATA (extracted_syllables Folders)

✓ Total extracted_syllables folders: 1
✓ Indices available: [1]
✓ These will be CONSOLIDATED into: extracted_syllables_consolidated/
   - extracted_syllables_1: 272 files

✓ Total files to consolidate: 272

✓ Helper functions loaded:
   - get_all_extracted_syllables_indices() → Get SEMUA extracted_syllables indices
   - consolidate_all_extracted_syllables() → Merge semua ke satu folder
   - get_next_augmented_index() → Get next SEQUENTIAL augmentation index


## 📊 Helper Functions untuk Analisis
Fungsi-fungsi utility untuk menganalisis dan membandingkan hasil augmentasi dari berbagai run.

**Fungsi Tersedia:**
- `summarize_augmented_run(aug_index)`: Tampilkan statistik detail untuk satu augmentation run
- `compare_augmentation_runs(run_1, run_2)`: Bandingkan hasil augmentasi antara dua run

**Kegunaan:**
- Verifikasi jumlah file yang dihasilkan per label
- Monitoring status kelengkapan dataset
- Membandingkan hasil augmentasi dari parameter berbeda

In [3]:
# ===== OPTIONAL: Helper Functions for Analysis =====

def summarize_augmented_run(aug_index):
    """
    Summarize augmentation run dengan statistik detail.
    
    Args:
        aug_index: Augmentation run index
    """
    aug_dir = os.path.join(augmentation_config.BASE_AUGMENTED_DIR, f"augmented_{aug_index}")
    
    if not os.path.exists(aug_dir):
        print(f"❌ Augmentation run {aug_index} tidak ditemukan")
        return
    
    print(f"\n📁 AUGMENTATION RUN #{aug_index}")
    print("="*60)
    
    total_files = 0
    label_summary = {}
    
    for label in sorted(augmentation_config.TARGET_LABELS):
        label_dir = os.path.join(aug_dir, label)
        count = len([f for f in os.listdir(label_dir) if f.endswith('.wav')]) if os.path.exists(label_dir) else 0
        label_summary[label] = count
        total_files += count
        
        status = "✅" if count >= augmentation_config.TARGET_SAMPLES_PER_LABEL else "⚠️"
        print(f"{status} {label:10s}: {count:3d} files")
    
    print("="*60)
    print(f"Total: {total_files} files")
    
    return label_summary


def compare_augmentation_runs(run_index_1, run_index_2=None):
    """
    Compare two augmentation runs.
    
    Args:
        run_index_1: First run index
        run_index_2: Second run index (optional)
    """
    summary_1 = summarize_augmented_run(run_index_1)
    
    if run_index_2:
        summary_2 = summarize_augmented_run(run_index_2)
        
        print("\n📊 COMPARISON")
        print("="*60)
        print(f"{'Label':10s} | {'Run {}'.format(run_index_1):10s} | {'Run {}'.format(run_index_2):10s} | Diff")
        print("-"*60)
        
        for label in sorted(augmentation_config.TARGET_LABELS):
            c1 = summary_1.get(label, 0)
            c2 = summary_2.get(label, 0)
            diff = c2 - c1
            diff_str = f"+{diff}" if diff >= 0 else f"{diff}"
            print(f"{label:10s} | {c1:10d} | {c2:10d} | {diff_str:6s}")

print("✓ Helper functions loaded: summarize_augmented_run(), compare_augmentation_runs()")


✓ Helper functions loaded: summarize_augmented_run(), compare_augmentation_runs()


## ⚙️ Configuration Display
Menampilkan semua parameter augmentasi yang akan digunakan untuk transform scraper data.

**ℹ️ Konfigurasi ini akan diterapkan pada:**
- Scraper run yang dipilih (via `source_run_index`)
- Menghasilkan augmentation run baru dengan karakteristik sesuai parameter

**Parameter Kunci:**
- **Target Samples per Label**: Jumlah target sampel per kategori suku kata untuk augmentasi
- **Pitch Shift Range**: Range perubahan pitch dalam semitone (variasi vokal)
- **Time Stretch Range**: Range perubahan durasi/tempo (variasi kecepatan bicara)
- **Target Duration**: Durasi standar untuk normalisasi audio
- **Sample Rate**: Sampling frequency audio (Hz) - harus konsisten dengan scraper
- **RMS Target**: Target volume level untuk normalisasi
- **Enable Flags**: Mengaktifkan/menonaktifkan fitur augmentasi tertentu

In [4]:
# ===== CONFIGURATION DISPLAY =====

print("\n" + "="*60)
print("AUGMENTATION CONFIGURATION")
print("="*60)

config_display = {
    "Target Samples per Label": augmentation_config.TARGET_SAMPLES_PER_LABEL,
    "Pitch Shift Range": f"{augmentation_config.PITCH_SHIFT_MIN} to {augmentation_config.PITCH_SHIFT_MAX} semitone",
    "Time Stretch Range": f"{augmentation_config.TIME_STRETCH_MIN} to {augmentation_config.TIME_STRETCH_MAX}",
    "Target Duration": f"{augmentation_config.TARGET_DURATION_SEC}s",
    "Sample Rate": f"{augmentation_config.SAMPLE_RATE} Hz",
    "Target RMS": augmentation_config.TARGET_RMS,
    "Pitch Shift Enabled": augmentation_config.ENABLE_PITCH_SHIFT,
    "Time Stretch Enabled": augmentation_config.ENABLE_TIME_STRETCH,
    "Duration Normalization": augmentation_config.ENABLE_DURATION_NORMALIZATION,
    "RMS Normalization": augmentation_config.ENABLE_RMS_NORMALIZATION,
}

for key, value in config_display.items():
    print(f"  {key:.<40s} {value}")

print(f"\n  Target Labels ({len(augmentation_config.TARGET_LABELS)}): {', '.join(augmentation_config.TARGET_LABELS)}")
print("="*60)



AUGMENTATION CONFIGURATION
  Target Samples per Label................ 200
  Pitch Shift Range....................... -4 to 4 semitone
  Time Stretch Range...................... 0.9 to 1.1
  Target Duration......................... 1.0s
  Sample Rate............................. 16000 Hz
  Target RMS.............................. 0.1
  Pitch Shift Enabled..................... True
  Time Stretch Enabled.................... True
  Duration Normalization.................. True
  RMS Normalization....................... True

  Target Labels (20): a, i, u, e, o, ma, mi, mu, me, mo, ba, bi, bu, be, bo, pa, pi, pu, pe, po


## 🚀 Main: 2-Stage Consolidation + Augmentation Pipeline
Mengeksekusi 2-stage pipeline:
1. **STAGE 1 - CONSOLIDATION**: Merge SEMUA extracted_syllables folders menjadi satu
2. **STAGE 2 - AUGMENTATION**: Augment data gabungan dengan single pass

### INPUT (STAGE 1):
- Membaca SEMUA `extracted_syllables_[X]` yang ada (X = 1, 2, 3, 5, ...)
- Setiap folder berisi label subdirectories dengan audio files

### CONSOLIDATION (STAGE 1):
- Merge semua files dari SETIAP extracted_syllables ke: `extracted_syllables_consolidated/`
- Organized by label: a/, ba/, be/, ma/, dll
- Hasil: Satu folder utuh dengan data gabungan

### AUGMENTATION (STAGE 2):
- Input: `extracted_syllables_consolidated/` (data hasil merge)
- Apply techniques: pitch shift, time stretch, duration normalization
- Output: `augmented_[Y]` (next sequential independent index)
- Hasil: Single augmented folder dengan complete augmented dataset

**Output Report:**
- ✅ STAGE 1: Consolidation status (sources, files per label, total)
- ✅ STAGE 2: Augmentation status (input, output, files per label)
- ✔️ Variable `consolidated_result` dan `current_aug_index` untuk analysis
- 📊 Complete summary dengan unified view


In [5]:
# ===== MAIN: 2-Stage Pipeline - Consolidation + Augmentation =====

extracted_indices = get_all_extracted_syllables_indices()

print("\n" + "="*70)
print("🎯 2-STAGE PIPELINE - Consolidation + Augmentation")
print("="*70)

if len(extracted_indices) == 0:
    print("\n❌ No extracted_syllables folders available!")
    print("   Run scraping.ipynb terlebih dahulu untuk generate data")
    consolidated_result = None
    current_aug_index = None
else:
    print(f"\n📥 INPUT: Found {len(extracted_indices)} extracted_syllables folder(s)")
    print(f"   Indices: {extracted_indices}")
    
    # ========== STAGE 1: CONSOLIDATION ==========
    print(f"\n{'─'*70}")
    print(f"STAGE 1️⃣  - CONSOLIDATION: Merge SEMUA extracted_syllables")
    print(f"{'─'*70}")
    
    consolidated_result = consolidate_all_extracted_syllables()
    
    if consolidated_result["status"] == "error":
        print(f"\n❌ Consolidation FAILED!")
        print(f"   Error: {consolidated_result.get('error', 'Unknown error')}")
        current_aug_index = None
    
    else:
        print(f"\n✅ CONSOLIDATION SUCCESS!")
        print(f"\n📊 Input Details:")
        print(f"   Source folders: {consolidated_result['input_indices']}")
        print(f"   Total files merged: {consolidated_result['total_files']}")
        
        print(f"\n📈 Files per label (consolidated):")
        for label in sorted(augmentation_config.TARGET_LABELS):
            count = consolidated_result['label_summary'].get(label, 0)
            status = "✅" if count > 0 else "⚠️"
            print(f"   {status} {label:10s}: {count:3d} files")
        
        consolidated_path = consolidated_result['consolidated_path']
        print(f"\n📁 Consolidated folder: {consolidated_path}")
        
        # ========== STAGE 2: AUGMENTATION ==========
        print(f"\n{'─'*70}")
        print(f"STAGE 2️⃣  - AUGMENTATION: Apply techniques to consolidated data")
        print(f"{'─'*70}")
        
        aug_index = get_next_augmented_index()
        aug_root = os.path.join(augmentation_config.BASE_AUGMENTED_DIR, f"augmented_{aug_index}")
        
        print(f"\n📋 Augmentation Config:")
        print(f"   Input: extracted_syllables_consolidated/ ({consolidated_result['total_files']} files)")
        print(f"   Output Index: augmented_#{aug_index}")
        print(f"   Target samples per label: {augmentation_config.TARGET_SAMPLES_PER_LABEL}")
        
        try:
            # Create output directories for each label
            print(f"\n⏳ Creating output directories...")
            for label in augmentation_config.TARGET_LABELS:
                label_dir = os.path.join(aug_root, label)
                os.makedirs(label_dir, exist_ok=True)
            
            # Copy original files dari consolidated folder
            print(f"📋 Copying consolidated files to augmented folder...")
            for label in augmentation_config.TARGET_LABELS:
                src_label_dir = os.path.join(consolidated_path, label)
                dst_label_dir = os.path.join(aug_root, label)
                
                if os.path.exists(src_label_dir):
                    for file in os.listdir(src_label_dir):
                        if file.endswith('.wav'):
                            src_file = os.path.join(src_label_dir, file)
                            dst_file = os.path.join(dst_label_dir, file)
                            shutil.copy2(src_file, dst_file)
            
            # Apply augmentation techniques
            print(f"\n🎵 Applying augmentation techniques...")
            print(f"   - Pitch shifting")
            print(f"   - Time stretching")
            print(f"   - Duration normalization")
            
            for label in augmentation_config.TARGET_LABELS:
                label_dir = os.path.join(aug_root, label)
                if os.path.exists(label_dir):
                    initial_count = len([f for f in os.listdir(label_dir) if f.endswith('.wav')])
                    
                    if initial_count < augmentation_config.TARGET_SAMPLES_PER_LABEL:
                        # Augment to target
                        audio_augmenter.augment_label_to_target(
                            label_dir,
                            label,
                            augmentation_config.TARGET_SAMPLES_PER_LABEL
                        )
            
            # Get final count
            final_files_per_label = {}
            total_final_files = 0
            for label in augmentation_config.TARGET_LABELS:
                label_dir = os.path.join(aug_root, label)
                if os.path.exists(label_dir):
                    count = len([f for f in os.listdir(label_dir) if f.endswith('.wav')])
                    final_files_per_label[label] = count
                    total_final_files += count
            
            # ========== SUCCESS REPORT ==========
            print(f"\n{'─'*70}")
            print(f"✅ AUGMENTATION SUCCESS!")
            print(f"{'─'*70}")
            
            print(f"\n📊 Output Details:")
            print(f"   Output folder: augmented_#{aug_index}/")
            print(f"   Total files generated: {total_final_files}")
            
            print(f"\n📈 Files per label (augmented):")
            for label in sorted(augmentation_config.TARGET_LABELS):
                count = final_files_per_label.get(label, 0)
                target = augmentation_config.TARGET_SAMPLES_PER_LABEL
                status = "✅" if count >= target else "⚠️"
                pct = (count / target * 100) if target > 0 else 0
                print(f"   {status} {label:10s}: {count:4d} files ({pct:5.1f}% of target)")
            
            print(f"\n✔️  Pipeline Complete!")
            print(f"   Consolidated path: {consolidated_path}")
            print(f"   Output path: {aug_root}")
            print(f"   Storage: {total_final_files} augmented files ready for training")
            
            print("="*70)
            
            current_aug_index = aug_index
        
        except Exception as e:
            print(f"\n❌ ERROR during augmentation!")
            print(f"   {str(e)}")
            import traceback
            traceback.print_exc()
            current_aug_index = None



🎯 2-STAGE PIPELINE - Consolidation + Augmentation

📥 INPUT: Found 1 extracted_syllables folder(s)
   Indices: [1]

──────────────────────────────────────────────────────────────────────
STAGE 1️⃣  - CONSOLIDATION: Merge SEMUA extracted_syllables
──────────────────────────────────────────────────────────────────────

✅ CONSOLIDATION SUCCESS!

📊 Input Details:
   Source folders: [1]
   Total files merged: 272

📈 Files per label (consolidated):
   ✅ a         :  70 files
   ✅ ba        :  17 files
   ✅ be        :  29 files
   ✅ bi        :   3 files
   ✅ bo        :   5 files
   ✅ bu        :   3 files
   ✅ e         :  13 files
   ✅ i         :  18 files
   ✅ ma        :  56 files
   ✅ me        :   3 files
   ⚠️ mi        :   0 files
   ⚠️ mo        :   0 files
   ✅ mu        :   3 files
   ✅ o         :   1 files
   ✅ pa        :  15 files
   ✅ pe        :   2 files
   ✅ pi        :   3 files
   ⚠️ po        :   0 files
   ✅ pu        :  13 files
   ✅ u         :  18 files

📁 Consoli

## 📈 Analysis: Verifikasi Hasil 2-Stage Pipeline
Menganalisis hasil STAGE 1 (Consolidation) dan STAGE 2 (Augmentation) dari main pipeline.

### STAGE 1 Results (Consolidation):
- **Input**: Semua `extracted_syllables_[X]` folders
- **Output**: `extracted_syllables_consolidated/` (merged data)
- **Verifikasi**: Files dikumpulkan dari semua sources
- **Status**: Apakah ada data loss atau duplicate?

### STAGE 2 Results (Augmentation):
- **Input**: `extracted_syllables_consolidated/` (dari STAGE 1)
- **Output**: `augmented_[Y]/` (final augmented data, next sequential index)
- **Verifikasi**: Augmentation mencapai target samples per label?
- **Status**: Siap untuk training atau masih kurang?

**Output Analysis Menampilkan:**
- 📊 **STAGE 1 Summary**: Label breakdown dari consolidated folder
  - Mana labels yang ada data dari multiple sources
  - Total files sebelum augmentation
- 📊 **STAGE 2 Summary**: Label breakdown dari augmented folder
  - ✅ Status "OK" = mencapai target
  - ⚠️ Status Warning = belum mencapai target
- 📄 **Perbandingan**: Before (consolidated) vs After (augmented)
- 🔍 **Statistik**: Total files, augmentation ratio, coverage

**Verifikasi Checklist:**
- ✓ Apakah STAGE 1 sukses merge semua extracted_syllables?
- ✓ Apakah semua labels memiliki data di consolidated folder?
- ✓ Apakah STAGE 2 augmentation sukses?
- ✓ Apakah semua labels mencapai TARGET_SAMPLES_PER_LABEL?
- ✓ Berapa total augmented files siap untuk training?

**Optional Next Steps:**
- Hapus `extracted_syllables_consolidated/` jika tidak diperlukan (hemat storage)
- Export augmented_[Y]/ untuk training pipeline
- Analisis audio quality dari augmented samples


In [6]:
# ===== ANALYSIS: Verifikasi 2-Stage Pipeline Results =====

print("\n" + "="*70)
print("📊 2-STAGE PIPELINE ANALYSIS - Results Verification")
print("="*70)

# Check if consolidation was successful
if 'consolidated_result' not in locals() or consolidated_result is None:
    print("\n⚠️  No consolidation results found!")
    print("   Run Main Pipeline cell first to generate consolidation + augmentation")
else:
    # ========== STAGE 1: CONSOLIDATION ANALYSIS ==========
    print(f"\n{'─'*70}")
    print(f"STAGE 1️⃣  - CONSOLIDATION RESULTS")
    print(f"{'─'*70}")
    
    if consolidated_result.get("status") == "success":
        print(f"\n✅ CONSOLIDATION SUCCESS!")
        print(f"\n📋 Input Sources:")
        input_indices = consolidated_result['input_indices']
        print(f"   Source extracted_syllables indices: {input_indices}")
        print(f"   Number of source folders: {len(input_indices)}")
        
        print(f"\n📊 Consolidated Data Summary:")
        print(f"   Total files merged: {consolidated_result['total_files']}")
        
        print(f"\n📈 Files per label (consolidated):")
        label_summary = consolidated_result['label_summary']
        for label in sorted(augmentation_config.TARGET_LABELS):
            count = label_summary.get(label, 0)
            status = "✅" if count > 0 else "❌"
            print(f"   {status} {label:10s}: {count:4d} files")
        
        print(f"\n📁 Location: {consolidated_result['consolidated_path']}")
    else:
        print(f"\n❌ CONSOLIDATION FAILED!")
        print(f"   Error: {consolidated_result.get('error', 'Unknown error')}")

# Check if augmentation was successful
if 'current_aug_index' not in locals() or current_aug_index is None:
    print(f"\n⚠️  No augmentation results found!")
    print("   Run Main Pipeline cell first to complete STAGE 2")
else:
    # ========== STAGE 2: AUGMENTATION ANALYSIS ==========
    print(f"\n{'─'*70}")
    print(f"STAGE 2️⃣  - AUGMENTATION RESULTS")
    print(f"{'─'*70}")
    
    aug_dir = os.path.join(augmentation_config.BASE_AUGMENTED_DIR, f"augmented_{current_aug_index}")
    
    if os.path.exists(aug_dir):
        print(f"\n✅ AUGMENTATION SUCCESSFUL!")
        print(f"\n📋 Augmentation Run #{current_aug_index}")
        
        # Get label statistics
        aug_label_stats = {}
        total_aug_files = 0
        
        print(f"\n📈 Files per label (augmented):")
        for label in sorted(augmentation_config.TARGET_LABELS):
            label_dir = os.path.join(aug_dir, label)
            count = 0
            if os.path.exists(label_dir):
                count = len([f for f in os.listdir(label_dir) if f.endswith('.wav')])
            
            aug_label_stats[label] = count
            total_aug_files += count
            
            target = augmentation_config.TARGET_SAMPLES_PER_LABEL
            pct = (count / target * 100) if target > 0 else 0
            
            if count >= target:
                status = "✅ OK"
            elif count > 0:
                status = f"⚠️  PARTIAL"
            else:
                status = "❌ EMPTY"
            
            print(f"   {status:12s} {label:10s}: {count:4d} / {target:4d} ({pct:6.1f}%)")
        
        print(f"\n📊 Overall Summary:")
        print(f"   Total augmented files: {total_aug_files}")
        print(f"   Labels with data: {sum(1 for c in aug_label_stats.values() if c > 0)}/{len(augmentation_config.TARGET_LABELS)}")
        print(f"   Output folder: augmented_#{current_aug_index}/")
        
        # ========== CONSOLIDATED vs AUGMENTED COMPARISON ==========
        if consolidated_result and consolidated_result.get("status") == "success":
            print(f"\n{'─'*70}")
            print(f"📊 BEFORE vs AFTER - Consolidation vs Augmentation")
            print(f"{'─'*70}")
            
            print(f"\n{'Label':<12} | {'Before (Consolidated)':<20} | {'After (Augmented)':<20} | {'Increase':<10}")
            print(f"{'-'*70}")
            
            consolidated_labels = consolidated_result['label_summary']
            total_before = consolidated_result['total_files']
            
            for label in sorted(augmentation_config.TARGET_LABELS):
                before = consolidated_labels.get(label, 0)
                after = aug_label_stats.get(label, 0)
                increase = after - before
                increase_pct = (increase / before * 100) if before > 0 else 0
                
                increase_str = f"+{increase} ({increase_pct:.0f}%)" if before > 0 else f"+{increase}"
                print(f"{label:<12} | {before:>8} files       | {after:>8} files       | {increase_str:<10}")
            
            print(f"{'-'*70}")
            print(f"{'TOTAL':<12} | {total_before:>8} files       | {total_aug_files:>8} files       | +{total_aug_files - total_before} ({((total_aug_files - total_before) / total_before * 100):.0f}%)")
        
        print(f"\n✔️  Pipeline Complete!")
        print(f"   Data ready for training at: {aug_dir}/")
        print("="*70)
    
    else:
        print(f"\n❌ Augmentation folder not found: {aug_dir}")



📊 2-STAGE PIPELINE ANALYSIS - Results Verification

──────────────────────────────────────────────────────────────────────
STAGE 1️⃣  - CONSOLIDATION RESULTS
──────────────────────────────────────────────────────────────────────

✅ CONSOLIDATION SUCCESS!

📋 Input Sources:
   Source extracted_syllables indices: [1]
   Number of source folders: 1

📊 Consolidated Data Summary:
   Total files merged: 272

📈 Files per label (consolidated):
   ✅ a         :   70 files
   ✅ ba        :   17 files
   ✅ be        :   29 files
   ✅ bi        :    3 files
   ✅ bo        :    5 files
   ✅ bu        :    3 files
   ✅ e         :   13 files
   ✅ i         :   18 files
   ✅ ma        :   56 files
   ✅ me        :    3 files
   ❌ mi        :    0 files
   ❌ mo        :    0 files
   ✅ mu        :    3 files
   ✅ o         :    1 files
   ✅ pa        :   15 files
   ✅ pe        :    2 files
   ✅ pi        :    3 files
   ❌ po        :    0 files
   ✅ pu        :   13 files
   ✅ u         :   18 files
